# Bài 7
Đây là notebook chứa mã nguồn đầy đủ của bài 7.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import pulp
import pyomo.environ as pyo
from scipy.optimize import linprog, minimize, milp, LinearConstraint, Bounds
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize as pymoo_minimize

from src.data_loader import get_data


In [ ]:
class VietnamDigitalProblem(ElementwiseProblem):
    def __init__(self):
        super().__init__(n_var=24, n_obj=4, n_ieq_constr=14, xl=np.zeros(24), xu=np.ones(24)*12000)
        # Beta matrix (6 vùng x 4 hạng mục)
        self.beta = np.array([
            [1.15, 0.85, 0.55, 1.30],
            [0.95, 1.25, 1.40, 1.05],
            [1.05, 0.95, 0.85, 1.15],
            [1.20, 0.75, 0.45, 1.35],
            [0.90, 1.30, 1.55, 1.00],
            [1.10, 0.85, 0.65, 1.25],
        ])
        self.e = np.array([0.42, 0.55, 0.48, 0.32, 0.62, 0.38])
        self.rho = np.array([0.18, 0.45, 0.28, 0.12, 0.52, 0.22])
        self.sig = np.array([0.32, 0.28, 0.30, 0.35, 0.25, 0.30])

    def _evaluate(self, x, out, *args, **kwargs):
        X = x.reshape(6, 4)
        # f1: max GDP gain => -sum(beta * X)
        f1 = -(self.beta * X).sum()
        # f2: Gini approximated by MAD
        sums = X.sum(axis=1)
        f2 = np.abs(sums - sums.mean()).mean()
        # f3: emissions proxy (AI related)
        f3 = (self.e * (X[:,0] + X[:,2])).sum()
        # f4: security proxy
        f4 = (self.sig * X[:,1]).sum()
        
        out["F"] = [f1, f2, f3, f4]
        
        # Constraints (C1: sum X <= 50000) -> sum X - 50000 <= 0
        g1 = X.sum() - 50000
        
        # C2: sum_j x_jr >= 5000 -> 5000 - sum_j x_jr <= 0
        g2 = 5000 - X.sum(axis=1) # array of 6
        
        # C3: sum_j x_jr <= 12000 -> sum_j x_jr - 12000 <= 0
        g3 = X.sum(axis=1) - 12000 # array of 6
        
        # C4: sum_r x_H,r >= 12000 -> 12000 - sum_r x_H,r <= 0
        g4 = 12000 - X[:,3].sum()
        
        out["G"] = [g1, *g2, *g3, g4]

In [ ]:
def solve_bai07(n_gen=30, pop_size=50, seed=42):
    try:
        problem = VietnamDigitalProblem()
        algorithm = NSGA2(pop_size=pop_size)
        res = pymoo_minimize(problem, algorithm, ('n_gen', n_gen), seed=seed, verbose=False)
        
        pareto_front = res.F
        pareto_solutions = res.X
        
        if pareto_front is not None and len(pareto_front) > 0:
            pareto_front[:, 0] = -pareto_front[:, 0]  # Convert f1 back to positive GDP
            n_pareto = len(pareto_front)
            
            # TOPSIS compromise
            PF = pareto_front.copy()
            PF_norm = PF / (np.sqrt((PF**2).sum(axis=0)) + 1e-9)
            w = np.array([0.40, 0.25, 0.20, 0.15])
            V = PF_norm * w
            ideal = np.array([V[:,0].max(), V[:,1].min(), V[:,2].min(), V[:,3].min()])
            anti_ideal = np.array([V[:,0].min(), V[:,1].max(), V[:,2].max(), V[:,3].max()])
            
            D_plus = np.sqrt(((V - ideal)**2).sum(axis=1))
            D_minus = np.sqrt(((V - anti_ideal)**2).sum(axis=1))
            C = D_minus / (D_plus + D_minus + 1e-9)
            best_idx = np.argmax(C)
            top_sol = PF[best_idx]
            
            # Opportunity cost (best GDP)
            best_gdp_idx = np.argmax(PF[:,0])
            best_gdp_sol = PF[best_gdp_idx]
            
            topsis_compromise = {
                'GDP': float(top_sol[0]), 'Equity_MAD': float(top_sol[1]), 
                'Emission': float(top_sol[2]), 'Security': float(top_sol[3])
            }
            
            sacrifice = {
                'Equity_MAD_pct': float((best_gdp_sol[1] - top_sol[1]) / (top_sol[1] + 1e-9) * 100),
                'Emission_pct': float((best_gdp_sol[2] - top_sol[2]) / (top_sol[2] + 1e-9) * 100),
                'Security_pct': float((best_gdp_sol[3] - top_sol[3]) / (top_sol[3] + 1e-9) * 100),
            }
            
            opportunity_cost = {
                'best_gdp_sol': {
                    'GDP': float(best_gdp_sol[0]), 'Equity_MAD': float(best_gdp_sol[1]), 
                    'Emission': float(best_gdp_sol[2]), 'Security': float(best_gdp_sol[3])
                },
                'sacrifice': sacrifice
            }
        else:
            pareto_front = np.array([])
            pareto_solutions = np.array([])
            n_pareto = 0
            topsis_compromise = {}
            opportunity_cost = {}
            
    except Exception:
        n_pareto = 0
        pareto_front = np.array([])
        pareto_solutions = np.array([])
        topsis_compromise = {}
        opportunity_cost = {}

    return {
        'n_pareto':        n_pareto,
        'history':         [],
        'pareto_front':    pareto_front.tolist(),
        'pareto_solutions': pareto_solutions.tolist(),
        'f1_gdp':    pareto_front[:, 0].tolist() if n_pareto > 0 else [],
        'f2_equity': pareto_front[:, 1].tolist() if n_pareto > 0 else [],
        'f3_env':    pareto_front[:, 2].tolist() if n_pareto > 0 else [],
        'f4_sec':    pareto_front[:, 3].tolist() if n_pareto > 0 else [],
        'topsis_compromise': topsis_compromise,
        'opportunity_cost': opportunity_cost
    }

In [ ]:
if __name__ == '__main__':
    res = solve_bai07()
    # In ra một số key để kiểm tra
    if isinstance(res, dict):
        print(res.keys())